# Следующее чтение: прогноз в момент записи
Цель: `t_next_read − t_write` в секундах для одного начального байтового адреса. Первое чтение после записи, включая случаи с промежуточной перезаписью. Прогноз обращения к адресу, не версии данных.

Сначала показан пилот на первых 1 000 000 событий `usr_1` и `prxy_1`, ниже — продолжение на первых 24 часах. Признаки из прошлого, хронологическое разделение 60/20/20 по длительности, два baseline и LightGBM-регрессия `log1p(seconds)`. Разметка не пересекает границы splits. Цензурированные записи исключены из оценки; coverage обязателен. Validation usr_1 пустой только в коротком пилоте; модель фиксированная, без подбора параметров.

In [ ]:
from pathlib import Path
import json, subprocess, sys
from IPython.display import Markdown, display
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'read_prediction').is_dir() and (p / 'pyproject.toml').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
OUTPUT = ROOT / 'results' / 'next-read-pilot'

## Воспроизведение
Сначала `uv sync --extra notebook` и выберите kernel `.venv/bin/python`. CLI: `uv run python -m read_prediction.pipeline --max-events 1000000 --output results/next-read-pilot`.
Для нового эксперимента задайте новый OUTPUT: существующие результаты не перезаписываются. Ячейка ниже по умолчанию только читает результаты.

In [ ]:
RUN_EXPERIMENT = False
if RUN_EXPERIMENT:
    subprocess.run([sys.executable, '-m', 'read_prediction.pipeline', '--max-events', '1000000', '--output', str(OUTPUT)], cwd=ROOT, check=True)

In [ ]:
record = json.loads((OUTPUT / 'results.json').read_text())
display(Markdown((OUTPUT / 'report.md').read_text()))

In [ ]:
for workload, result in record['results'].items():
    print(workload, 'events:', result['source']['loaded_events'], 'duration, s:', round(result['duration_seconds'], 2))
    scores = result['metrics']['test']
    if scores['lightgbm'] is None:
        print('  Test metrics unavailable: no completed intervals')
        continue
    model_mae = scores['lightgbm']['mae_seconds']
    for baseline in ('global_median', 'address_median'):
        baseline_mae = scores[baseline]['mae_seconds']
        improvement = 100 * (baseline_mae - model_mae) / baseline_mae if baseline_mae else float('nan')
        print(f'  MAE improvement vs {baseline}: {improvement:.1f}%')
print('Process peak RSS, MiB:', round(record['process_peak_rss_mib_linux'], 1))

## Интерпретация
Сравниваем модель с обоими baseline, а не только с постоянным прогнозом. Метрики относятся только к записям с чтением до конца split. Длинные интервалы чаще исключаются; при небольшом coverage оценка не описывает все записи. Начальные фрагменты не доказывают качество на полной неделе или ускорение SSD. Подробный протокол и команды — `README.md`.

## Продолжение: первые 24 часа
Параметры модели и правила разметки сохранены. Проверка на более длинном периоде дала разные результаты: модель лучше простой медианы на usr_1, но хуже на prxy_1. Ниже показан сохранённый суточный результат без нового обучения.

In [ ]:
EXTENDED = ROOT / 'results' / 'next-read-24h'
if (EXTENDED / 'results.json').exists():
    display(Markdown((EXTENDED / 'summary.md').read_text()))
    display(Markdown((EXTENDED / 'report.md').read_text()))
else:
    print('Суточный результат ещё не сохранён. Команда запуска — в README.md.')

## Диагностика prxy_1: только train и validation
Проверены накопленные счётчики и функция обучения. Выбор нового варианта сделан по validation; старый test повторно не оценивается. Качество на независимом будущем участке ещё не установлено.

In [ ]:
DIAGNOSIS = ROOT / 'results' / 'prxy-validation-diagnosis'
if (DIAGNOSIS / 'summary.md').exists():
    display(Markdown((DIAGNOSIS / 'summary.md').read_text()))
else:
    print('Диагностика ещё не выполнена.')

## Будущий период: вторые сутки prxy_1
Модель выбрана раньше, заморожена и не переобучается. Проверка 24–48h сравнивает её с теми же baseline, рассчитанными только по исходному обучению. Это новый период, не повторная настройка на прежнем test.

In [ ]:
DAY2 = ROOT / 'results' / 'prxy-day2'
if (DAY2 / 'summary.md').exists():
    display(Markdown((DAY2 / 'summary.md').read_text()))
else:
    print('Проверка вторых суток ещё не выполнена.')